# multilingual-e5-base - Cross-Lingual Text Embeddings on Amazon SageMaker

Deploys [intfloat/multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) from AWS
Marketplace as a SageMaker endpoint inside **your own AWS account**. Your text never
leaves your VPC and there are no external API calls or token limits.

**Model facts** (from the official model card): 768-dimensional vectors, mean pooling,
supports 100 languages, MIT licence.

Vectors are mean-pooled and L2-normalised, so cosine similarity is a plain dot product.

**Important:** multilingual-e5 requires a task prefix on every input.
Use `query: ` for search queries and `passage: ` for documents/passages.
Omitting the prefix degrades retrieval quality significantly.

**When to choose multilingual-e5-base:**
- Multilingual semantic search across 100 languages at **$0.08/hr**
- Cross-lingual retrieval (query in English, passages in Spanish, etc.)
- Multilingual clustering, classification, and sentence similarity

## 1. Prerequisites

1. Subscribe to the product in AWS Marketplace.
2. Copy the **model package ARN** shown on the product's launch page for your Region.
3. Run this notebook with a role that has `AmazonSageMakerFullAccess`.

In [ ]:
!pip install -qU sagemaker boto3

In [ ]:
import json

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN from the product's launch page for YOUR Region.
MODEL_PACKAGE_ARN = "<paste-model-package-arn-here>"

INSTANCE_TYPE = "ml.m5.xlarge"  # the recommended real-time instance
ENDPOINT_NAME = "multilingual-e5-base"

session = sagemaker.Session()
role = sagemaker.get_execution_role()
print("region:", session.boto_region_name)

## 2. Deploy a real-time endpoint

Takes roughly 6-9 minutes. The endpoint bills **$0.08/hr** while it exists,
so do not skip section 6.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=session,
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)
print("endpoint ready:", ENDPOINT_NAME)

## 3. Embed text

The endpoint accepts `application/json` shaped `{"inputs": "..."}` for a single string,
or `{"inputs": ["...", "..."]}` for a batch. It returns
`{"embeddings": [[...]], "dim": 768}`.

**Always prefix inputs:**
- `query: ` — for search queries
- `passage: ` — for documents and passages

In [ ]:
runtime = boto3.client("sagemaker-runtime")


def embed(texts):
    """Return a list of 768-dimension mean-pooled L2-normalised embedding vectors."""
    if isinstance(texts, str):
        texts = [texts]
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": texts}),
    )
    result = json.loads(response["Body"].read())
    if "embeddings" not in result:
        raise ValueError(f"Unexpected response format: {result}")
    return result["embeddings"]


# Single query embedding -- note the 'query: ' prefix
query_text = "query: What is the capital of France?"
query_vector = embed(query_text)[0]
print("dimensions:", len(query_vector))  # should be 768
print("first 8 values:", [round(v, 5) for v in query_vector[:8]])

## 4. Multilingual examples: English, Spanish, and French

multilingual-e5-base shares a single embedding space across 100 languages.
Semantically equivalent sentences in different languages produce similar vectors,
enabling cross-lingual retrieval without translation.

In [ ]:
# Passages in three languages -- note the 'passage: ' prefix
multilingual_passages = [
    # English
    "passage: Paris is the capital and largest city of France.",
    "passage: The Eiffel Tower was built in 1889 and stands 330 metres tall.",
    # Spanish
    "passage: París es la capital y la ciudad más grande de Francia.",
    "passage: La Torre Eiffel fue construida en 1889 y tiene 330 metros de altura.",
    # French
    "passage: Paris est la capitale et la plus grande ville de France.",
    "passage: La Tour Eiffel a été construite en 1889 et mesure 330 mètres de hauteur.",
    # Unrelated passage for contrast
    "passage: Amazon SageMaker is a managed machine learning platform.",
]

passage_vectors = embed(multilingual_passages)
print(f"Embedded {len(passage_vectors)} multilingual passages ({len(passage_vectors[0])} dims each)")

## 5. Cross-lingual similarity

Query in English; retrieve across English, Spanish, and French passages.
The model's shared multilingual space means semantically matching passages
score highly regardless of their language.

In [ ]:
def cosine_similarity(a, b):
    """Dot product of two L2-normalised vectors equals cosine similarity."""
    return sum(x * y for x, y in zip(a, b))


# Cross-lingual query: English query against passages in three languages
query = "query: What is the capital of France?"
query_vec = embed(query)[0]

print(f"Query: '{query}'\n")
print("Cross-lingual retrieval scores (higher = more relevant):\n")

scored = [
    (cosine_similarity(query_vec, pvec), passage)
    for pvec, passage in zip(passage_vectors, multilingual_passages)
]

for score, passage in sorted(scored, reverse=True):
    print(f"  {score:.4f}  {passage}")

print("\nNote: English, Spanish, and French descriptions of Paris all score")
print("much higher than the unrelated SageMaker passage.")

## 6. Clean up

Delete the endpoint when you are done. It bills $0.08/hr for as long as it is running.

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()
print("deleted:", ENDPOINT_NAME)